In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


# CONVERT RDF/BOT GRAPH INTO A TOPOLOGIC GRAPH

### This script reads in an RDF/BOT file and converts it into a topologic graph. This is an experimental feature and we welcome your feedback on it.

#### This script requires topologicpy version 0.5.7 or newer

## Import Libraries

In [ ]:
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.TGraph import TGraph
from topologicpy.Dictionary import Dictionary
from topologicpy.Plotly import Plotly
from topologicpy.Helper import Helper

## Check the topologicpy version

In [ ]:
print("This script requires topologicpy version 0.9.39 or newer.")
print(Helper.Version())

## Specify the path to the RDF/BOT file

In [ ]:
path = r"C:\Users\sarwj\Downloads\house.ttl" # Change Me!

## Import the RDF/BOT file into a Topologic Graph

In [ ]:
# inlcudeContext adds the Stories, Building, and Site to the graph
g = TGraph.ByBOTPath(path, includeContext=True)
print(g)

## Compile all the types in the graph for colour-coding

In [ ]:
types = []
vertices = TGraph.Vertices(g)
root_vertex = None
for v in vertices:
    d = _record_dictionary(v)
    v_type = Dictionary.ValueAtKey(d, 'type')
    if v_type.lower() == "site":
        root_vertex = v
    types.append(Dictionary.ValueAtKey(d, 'type'))
types = list(set(types))
types.sort()

## Retrieve all the 3D Geometries from the graph

In [ ]:
vertices = TGraph.Vertices(g)
geometries = []
for v in vertices:
    d = _record_dictionary(v)
    keys = Dictionary.Keys(d)
    for key in keys:
        if key.lower() == "brep":
            brepString = Dictionary.ValueAtKey(d, key)
            if brepString is not None:
                print("BREP String:", brepString)
                geom = Topology.ByBREPString(brepString)
                d = Dictionary.RemoveKey(d, "breptype")
                d = Dictionary.RemoveKey(d, "breptypestring")
                geom = Topology.SetDictionary(geom, d)
                geometries.append(geom)

In [ ]:
print(geometries)

## Draw the building and the graph (3D layout, colour-coded)

In [ ]:
data = []
for geom in geometries:
    d = _record_dictionary(geom)
    type = Dictionary.ValueAtKey(d,"type")
    if type.lower() == "wall" or type.lower() == "slab" or type.lower() == "roof":
        data += Plotly.DataByTopology(geom, faceOpacity=0)
    elif type.lower() == "door":
        data += Plotly.DataByTopology(geom, faceColor="brown", faceOpacity=0.2)
    elif type.lower() == "window":
        data += Plotly.DataByTopology(geom, faceColor="lightblue", faceOpacity=0.8)
    elif type.lower() == "element":
        data += Plotly.DataByTopology(geom, faceColor="yellow", faceOpacity=0.1)
data +=Plotly.DataByGraph(g,vertexLabelKey="label", vertexGroupKey='type', vertexGroups=types, vertexSize=4)
figure = Plotly.FigureByData(data, backgroundColor="white")
Plotly.Show(figure)

## Draw only the graph (3D layout, colour-coded)

In [ ]:
Topology.Show(g, vertexLabelKey="label", vertexGroupKey='type', vertexGroups=types, vertexSize=4)

## Draw a flattened version of the graph (spring layout, colour-coded)

In [ ]:
flat_g = TGraph.Reshape(g, k=0.1)
Topology.Show(flat_g, vertexLabelKey="label", vertexGroupKey='type', vertexGroups=types, vertexSize=4)